---
title: 03 Scrape YouTube Metadata With yt-dlp
execute:
  eval: false
---

This notebook demonstrates how to build a video-level metadata table for the videos observed in donated YouTube watch histories.

It reads unique `video_id` values from `outputs/tables/video_histories.csv`, uses `yt_dlp` to request public YouTube metadata, and writes one metadata row per unique video to `outputs/tables/meta_data.csv`.
**Public-data note.** The video IDs included in this public repository are scrambled fake IDs. They preserve the shape of the workflow but cannot be scraped from YouTube. To run this notebook for a real project, replace the input `video_id` values with real YouTube IDs and replace `YOUR_REGISTERED_TOKEN` with your registered identifying token.



## What This Notebook Produces

The main output is `outputs/tables/meta_data.csv`.

Each row represents one unique YouTube video ID. Metadata collection is attempted for every unique `video_id` in `video_histories.csv`. When `yt_dlp` cannot retrieve metadata, the row is still kept and the raw error message is stored in `error`.

## Dependency Check

This notebook requires `yt_dlp`. The import check below does not install anything automatically; it prints the command to run if the package is missing.

In [1]:
from pathlib import Path
from importlib.util import find_spec
import json
import time

import pandas as pd

if find_spec("yt_dlp") is None:
    print('yt_dlp is not installed in this kernel.')
    print('Install it with: %pip install yt-dlp')
else:
    import yt_dlp as yt
    print(f"yt_dlp available: {yt.version.__version__}")

## Locate The Project And Configure The Scraper

Run this notebook from the project root. Replace `YOUR_REGISTERED_TOKEN` with the registered identifying token or User-Agent string agreed for your project.

In [2]:
PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "_quarto.yml").exists():
    raise FileNotFoundError(
        "Run this notebook from the project root: cd youtube-donation-dsa-method-public, then open JupyterLab or run quarto render."
    )
OUTPUT_DIR = PROJECT_ROOT / "outputs" / "tables"
VIDEO_HISTORIES_PATH = OUTPUT_DIR / "video_histories.csv"
META_DATA_PATH = OUTPUT_DIR / "meta_data.csv"
SUBTITLES_DIR = PROJECT_ROOT / "outputs" / "subtitles_tmp"

IDENTIFYING_USER_AGENT = "YOUR_REGISTERED_TOKEN"
SLEEP_SECONDS_BETWEEN_REQUESTS = 0.0
MAX_VIDEOS_FOR_TEST = None
SUBTITLE_LANGUAGES = ["en"]

print(f"Project folder: {PROJECT_ROOT.name}")
print(f"Input table: {VIDEO_HISTORIES_PATH.relative_to(PROJECT_ROOT).as_posix()}")
print(f"Output table: {META_DATA_PATH.relative_to(PROJECT_ROOT).as_posix()}")

## Load Unique Video IDs

The metadata table is video-level, not watch-event-level. We therefore deduplicate the `video_id` values from `video_histories.csv` before scraping.

In [3]:
if not VIDEO_HISTORIES_PATH.exists():
    raise FileNotFoundError(
        f"Missing input table: {VIDEO_HISTORIES_PATH.relative_to(PROJECT_ROOT).as_posix()}"
    )

video_histories = pd.read_csv(VIDEO_HISTORIES_PATH, dtype={"video_id": "string"})

if "video_id" not in video_histories.columns:
    raise ValueError("video_histories.csv must contain a video_id column.")

unique_video_ids = sorted(video_histories["video_id"].dropna().unique())

if MAX_VIDEOS_FOR_TEST is not None:
    unique_video_ids = unique_video_ids[:MAX_VIDEOS_FOR_TEST]

pd.Series(
    {
        "watch_rows": len(video_histories),
        "unique_video_ids_to_scrape": len(unique_video_ids),
        "max_videos_for_test": MAX_VIDEOS_FOR_TEST,
    }
)

## Metadata Schema

The fields below are a compact CSV version of the original scraper schema. The `error` column is intentionally part of the schema because unavailable, private, removed, or age-restricted videos can be analytically informative.

In [4]:
CORE_METADATA_FIELDS = [
    "id",
    "title",
    "description",
    "channel_id",
    "channel",
    "uploader",
    "duration",
    "view_count",
    "like_count",
    "comment_count",
    "channel_follower_count",
    "upload_date",
    "timestamp",
    "webpage_url",
    "categories",
    "tags",
    "language",
    "resolution",
    "width",
    "height",
    "fps",
    "aspect_ratio",
]

SUBTITLE_FIELDS = [
    "requested_subtitles",
    "subtitles",
    "available_manual_subtitle_languages",
    "available_auto_caption_languages",
    "downloaded_subtitle_languages",
    "downloaded_subtitle_file_count",
]

METADATA_FIELDS = CORE_METADATA_FIELDS + SUBTITLE_FIELDS + ["metadata_status", "error"]

METADATA_FIELDS

## Scraper Helpers

The notebook requests manually provided subtitles and automatic captions in English, stores downloaded VTT files temporarily, reads their text into the metadata row, deletes the temporary files, and records which subtitle languages were available and downloaded.

In [5]:
class DummyLogger:
    def debug(self, msg):
        pass

    def warning(self, msg):
        pass

    def error(self, msg):
        pass


def validate_scraper_config():
    if IDENTIFYING_USER_AGENT == "YOUR_REGISTERED_TOKEN":
        raise ValueError(
            'Replace IDENTIFYING_USER_AGENT = "YOUR_REGISTERED_TOKEN" with your registered token before scraping.'
        )
    if find_spec("yt_dlp") is None:
        raise ImportError(
            'yt_dlp is not installed. Run: %pip install "yt-dlp[default,curl-cffi]"'
        )


def as_json_text(value):
    if value is None:
        return None
    if isinstance(value, (list, dict)):
        return json.dumps(value, ensure_ascii=False, default=str)
    return value


def comma_join_keys(value):
    if isinstance(value, dict):
        return ",".join(sorted(value.keys()))
    return None


def empty_metadata_row(video_id):
    row = {field: None for field in METADATA_FIELDS}
    row["id"] = video_id
    row["metadata_status"] = "pending"
    row["downloaded_subtitle_file_count"] = 0
    return row

In [6]:
def get_metadata(video_id, quiet=True, no_warnings=True):
    import yt_dlp as yt

    url = f"https://www.youtube.com/watch?v={video_id}"
    ydl_opts = {
        "quiet": quiet,
        "no_warnings": no_warnings,
        "logger": DummyLogger(),
        "skip_download": True,
        "writesubtitles": True,
        "writeautomaticsub": True,
        "subtitlesformat": "vtt",
        "subtitleslangs": SUBTITLE_LANGUAGES,
        "outtmpl": str(SUBTITLES_DIR / video_id),
        "http_headers": {"User-Agent": IDENTIFYING_USER_AGENT},
    }

    row = empty_metadata_row(video_id)

    try:
        with yt.YoutubeDL(ydl_opts) as ydl:
            # download=True allows subtitle files to be written; skip_download=True prevents video download.
            info = ydl.extract_info(url, download=True)

        for field in CORE_METADATA_FIELDS:
            row[field] = as_json_text(info.get(field))

        manual_subtitles = info.get("subtitles") or {}
        automatic_captions = info.get("automatic_captions") or {}
        requested_subtitles = info.get("requested_subtitles") or {}

        row["available_manual_subtitle_languages"] = comma_join_keys(manual_subtitles)
        row["available_auto_caption_languages"] = comma_join_keys(automatic_captions)
        row["requested_subtitles"] = as_json_text(requested_subtitles)

        subtitle_texts = []
        downloaded_languages = []
        for language, subtitle_info in requested_subtitles.items():
            subtitle_file = subtitle_info.get("filepath") if isinstance(subtitle_info, dict) else None
            if subtitle_file and Path(subtitle_file).exists():
                subtitle_text = Path(subtitle_file).read_text(encoding="utf-8")
                subtitle_texts.append(f"WEBVTT_LANGUAGE: {language}\n{subtitle_text}")
                downloaded_languages.append(language)
                try:
                    Path(subtitle_file).unlink()
                except OSError as exc:
                    print(f"Warning: failed to delete subtitle file {subtitle_file}: {exc}")

        row["subtitles"] = "\n\n".join(subtitle_texts) if subtitle_texts else None
        row["downloaded_subtitle_languages"] = ",".join(downloaded_languages) if downloaded_languages else None
        row["downloaded_subtitle_file_count"] = len(downloaded_languages)
        row["metadata_status"] = "ok"

    except Exception as exc:
        # Keep the raw yt_dlp message. These messages often explain why metadata was unavailable.
        row["error"] = str(exc)
        row["metadata_status"] = "error"

    return row

## Run The Scrape

This cell performs live requests. It will stop until `IDENTIFYING_USER_AGENT` is changed from `YOUR_REGISTERED_TOKEN`. The default delay between requests is `0` because this notebook assumes the project has permission to scrape; increase `SLEEP_SECONDS_BETWEEN_REQUESTS` if your access conditions require slower pacing.

In [7]:
validate_scraper_config()
SUBTITLES_DIR.mkdir(parents=True, exist_ok=True)

metadata_rows = []
total_ids = len(unique_video_ids)

for index, video_id in enumerate(unique_video_ids, start=1):
    print(f"Scraping {index}/{total_ids}: {video_id}")
    metadata_rows.append(get_metadata(video_id))

    if SLEEP_SECONDS_BETWEEN_REQUESTS and index < total_ids:
        time.sleep(SLEEP_SECONDS_BETWEEN_REQUESTS)

meta_data = pd.DataFrame(metadata_rows, columns=METADATA_FIELDS)
meta_data.to_csv(META_DATA_PATH, index=False)

META_DATA_PATH.relative_to(PROJECT_ROOT).as_posix()

## Diagnostics

After scraping, these diagnostics summarize coverage and errors. Error rows are not failures of the notebook; they are data about videos that could not be retrieved or whose subtitles could not be downloaded.

In [8]:
diagnostics = {
    "unique_video_ids_requested": len(unique_video_ids),
    "metadata_rows_written": len(meta_data),
    "unique_metadata_ids": meta_data["id"].nunique(),
    "rows_with_errors": int(meta_data["error"].notna().sum()),
    "rows_with_subtitles": int(meta_data["subtitles"].notna().sum()),
}

pd.Series(diagnostics)

In [9]:
error_summary = (
    meta_data.loc[meta_data["error"].notna(), "error"]
    .value_counts()
    .rename_axis("error")
    .reset_index(name="rows")
)

error_summary.head(10)

## Result

The stored metadata table is keyed by `id`, which is the YouTube `video_id`. It can now be joined back to `video_histories.csv` in the next notebook.

In [10]:
print(f"meta_data.csv ({len(meta_data)} rows)")
display(meta_data.head(5))